# Monolithic EEGMAT Pipeline\nAll code in one notebook.

In [ ]:
# ========================================
# eegmat_preprocessing.py
# ========================================

"""
eegmat_preprocessing.py — Best-practice preprocessing pipeline for the
PhysioNet EEGMAT dataset (rest vs. mental-arithmetic EEG).

Order of operations follows MNE-Python best practice:
  1. Load EDF + fix channel names ('EEG Fp1' → 'Fp1') + drop non-EEG
  2. Set 10-20 montage + re-reference to common average
  3. Band-pass filter 1-40 Hz (zero-phase FIR) on CONTINUOUS signal
  4. Notch filter at 50 Hz (line noise)
  5. Epoch into fixed-length 4-second windows
  6. Automatic bad-epoch rejection/repair via autoreject
  7. Return CLEAN, RAW-AMPLITUDE epochs (no z-scoring)

Usage:
    from eegmat_preprocessing import preprocess_subject, preprocess_all_subjects

    # Single subject
    epochs_rest, epochs_task = preprocess_subject(
        rest_edf_path='Subject00_1.edf',
        task_edf_path='Subject00_2.edf',
    )

    # All subjects
    X, y, groups, ch_names, info = preprocess_all_subjects('path/to/data/')
"""

import os
import re
import numpy as np
import mne

# Try importing autoreject; fall back to simple threshold if unavailable
try:
    from autoreject import AutoReject
    HAS_AUTOREJECT = True
except ImportError:
    HAS_AUTOREJECT = False
    print("Warning: autoreject not installed. Using simple amplitude threshold "
          "for artifact rejection (pip install autoreject for better results).")

# ── Configuration ──────────────────────────────────────────────────────────
CONFIG = {
    'l_freq': 1.0,           # high-pass cutoff (Hz) — removes slow drift
    'h_freq': 40.0,          # low-pass cutoff (Hz) — removes muscle/HF noise
    'notch_freq': 50.0,      # line noise (50 Hz in Europe/Asia; 60 Hz in Americas)
    'epoch_length': 4.0,     # seconds per epoch
    'epoch_overlap': 0.0,    # non-overlapping epochs (avoids correlated-epoch inflation)
    'montage': 'standard_1020',
    'reject_threshold': 150e-6,  # fallback: 150 µV peak-to-peak if autoreject unavailable
}

# Channels to drop from the raw EDF files (non-EEG)
DROP_CHANNELS = ['A2-A1', 'ECG']

# Standard 10-20 channel name mapping for the EEGMAT dataset
# EDF headers use 'EEG Fp1' format; we need bare '10-20' names for the montage
CHANNEL_RENAME = {
    'EEG Fp1': 'Fp1', 'EEG Fp2': 'Fp2',
    'EEG F3': 'F3',   'EEG F4': 'F4',
    'EEG F7': 'F7',   'EEG F8': 'F8',
    'EEG T3': 'T3',   'EEG T4': 'T4',
    'EEG C3': 'C3',   'EEG C4': 'C4',
    'EEG T5': 'T5',   'EEG T6': 'T6',
    'EEG P3': 'P3',   'EEG P4': 'P4',
    'EEG O1': 'O1',   'EEG O2': 'O2',
    'EEG Fz': 'Fz',   'EEG Cz': 'Cz',
    'EEG Pz': 'Pz',
    'EEG A2-A1': 'A2-A1',
    'ECG ECG': 'ECG',
}


def _fix_channel_names(raw):
    """
    Rename EDF channel labels ('EEG Fp1' → 'Fp1') and drop non-EEG channels.
    Returns the modified raw object in-place.
    """
    # Build rename mapping for channels present in this recording
    rename_map = {}
    for old_name in raw.ch_names:
        if old_name in CHANNEL_RENAME:
            rename_map[old_name] = CHANNEL_RENAME[old_name]
        else:
            # Generic fallback: strip 'EEG ' prefix if present
            stripped = re.sub(r'^EEG\s+', '', old_name).strip()
            if stripped != old_name:
                rename_map[old_name] = stripped

    if rename_map:
        raw.rename_channels(rename_map)

    # Drop non-EEG channels if present
    to_drop = [ch for ch in DROP_CHANNELS if ch in raw.ch_names]
    if to_drop:
        raw.drop_channels(to_drop)

    # Set all remaining channels as EEG type
    raw.set_channel_types({ch: 'eeg' for ch in raw.ch_names})

    return raw


def load_and_filter(edf_path, config=None):
    """
    Steps 1-4: load EDF, fix channel names, set montage, re-reference, filter.
    All filtering is done on the CONTINUOUS signal (before epoching) to avoid
    edge artifacts that occur when filtering short segments.

    Parameters
    ----------
    edf_path : str
        Path to an EDF file.
    config : dict, optional
        Override default CONFIG parameters.

    Returns
    -------
    raw : mne.io.Raw
        Preprocessed continuous EEG data.
    """
    if config is None:
        config = CONFIG

    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)

    # Fix channel names and drop non-EEG channels
    _fix_channel_names(raw)

    # Set 10-20 montage (needed for topomaps later)
    montage = mne.channels.make_standard_montage(config['montage'])
    raw.set_montage(montage, on_missing='warn', match_case=False)

    # Common-average re-reference
    raw.set_eeg_reference('average', projection=False, verbose=False)

    # Band-pass filter (zero-phase FIR) — BEFORE epoching per MNE best practice
    raw.filter(
        l_freq=config['l_freq'],
        h_freq=config['h_freq'],
        method='fir',
        phase='zero',
        verbose=False,
    )

    # Notch filter for residual 50 Hz line noise (harmless if already clean)
    raw.notch_filter(freqs=config['notch_freq'], verbose=False)

    return raw


def epoch_raw(raw, config=None):
    """
    Step 5: create fixed-length epochs from the filtered continuous signal.

    Parameters
    ----------
    raw : mne.io.Raw
        Filtered continuous EEG data.
    config : dict, optional
        Override default CONFIG parameters.

    Returns
    -------
    epochs : mne.Epochs
        Fixed-length epochs, preloaded.
    """
    if config is None:
        config = CONFIG

    epochs = mne.make_fixed_length_epochs(
        raw,
        duration=config['epoch_length'],
        overlap=config['epoch_overlap'],
        preload=True,
        verbose=False,
    )
    return epochs


def clean_epochs(epochs, config=None, random_state=42):
    """
    Step 6: automatic artifact rejection/repair.

    Uses autoreject if available (learns per-channel thresholds via CV).
    Falls back to simple peak-to-peak amplitude threshold otherwise.

    Parameters
    ----------
    epochs : mne.Epochs
        Raw epochs to clean.
    config : dict, optional
        Override default CONFIG parameters.
    random_state : int
        Random seed for autoreject reproducibility.

    Returns
    -------
    epochs_clean : mne.Epochs
        Artifact-cleaned epochs.
    n_rejected : int
        Number of epochs rejected.
    """
    if config is None:
        config = CONFIG

    n_before = len(epochs)

    if HAS_AUTOREJECT:
        ar = AutoReject(random_state=random_state, n_jobs=1, verbose=False)
        epochs_clean = ar.fit_transform(epochs)
    else:
        # Fallback: reject epochs where any channel exceeds threshold
        reject = dict(eeg=config['reject_threshold'])
        epochs_clean = epochs.copy().drop_bad(reject=reject, verbose=False)

    n_after = len(epochs_clean)
    n_rejected = n_before - n_after

    return epochs_clean, n_rejected


def preprocess_subject(rest_edf_path, task_edf_path, config=None):
    """
    Full preprocessing pipeline for one subject.

    Parameters
    ----------
    rest_edf_path : str
        Path to the rest (baseline) EDF file (*_1.edf).
    task_edf_path : str
        Path to the task (arithmetic) EDF file (*_2.edf).
    config : dict, optional
        Override default CONFIG parameters.

    Returns
    -------
    epochs_rest : mne.Epochs
        Clean rest epochs (raw amplitude, NOT z-scored).
    epochs_task : mne.Epochs
        Clean task epochs (raw amplitude, NOT z-scored).
    info : dict
        Preprocessing summary with rejection counts.
    """
    if config is None:
        config = CONFIG

    # Load and filter continuous data
    raw_rest = load_and_filter(rest_edf_path, config)
    raw_task = load_and_filter(task_edf_path, config)

    # Epoch
    epochs_rest_raw = epoch_raw(raw_rest, config)
    epochs_task_raw = epoch_raw(raw_task, config)

    # Clean
    epochs_rest, n_rej_rest = clean_epochs(epochs_rest_raw, config)
    epochs_task, n_rej_task = clean_epochs(epochs_task_raw, config)

    info = {
        'rest_epochs_total': len(epochs_rest_raw),
        'rest_epochs_clean': len(epochs_rest),
        'rest_rejected': n_rej_rest,
        'task_epochs_total': len(epochs_task_raw),
        'task_epochs_clean': len(epochs_task),
        'task_rejected': n_rej_task,
        'ch_names': epochs_rest.ch_names,
        'sfreq': epochs_rest.info['sfreq'],
    }

    return epochs_rest, epochs_task, info


def discover_subject_pairs(data_dir):
    """
    Auto-discover SubjectXX_1.edf / SubjectXX_2.edf pairs in a directory.

    Returns
    -------
    pairs : list of (subject_id, rest_path, task_path)
        Sorted by subject number.
    """
    files = os.listdir(data_dir)
    pattern = re.compile(r'^Subject(\d+)_(\d)\.edf$', re.IGNORECASE)

    subjects = {}
    for f in files:
        m = pattern.match(f)
        if m:
            subj_num = int(m.group(1))
            file_type = int(m.group(2))  # 1 = rest, 2 = task
            if subj_num not in subjects:
                subjects[subj_num] = {}
            subjects[subj_num][file_type] = os.path.join(data_dir, f)

    # Only include subjects with both rest and task files
    pairs = []
    for subj_num in sorted(subjects.keys()):
        if 1 in subjects[subj_num] and 2 in subjects[subj_num]:
            pairs.append((subj_num, subjects[subj_num][1], subjects[subj_num][2]))
        else:
            print(f"Warning: Subject{subj_num:02d} missing rest or task file, skipping.")

    return pairs


def preprocess_all_subjects(data_dir, config=None, subject_range=None):
    """
    Preprocess all subjects in a directory.

    Parameters
    ----------
    data_dir : str
        Path to the directory containing EDF files.
    config : dict, optional
        Override default CONFIG parameters.
    subject_range : tuple of (start, end), optional
        Process only subjects in range [start, end] inclusive.

    Returns
    -------
    X : np.ndarray, shape (n_total_epochs, n_channels, n_times)
        Raw-amplitude epoch data (NOT z-scored).
    y : np.ndarray, shape (n_total_epochs,)
        Labels: 0 = rest, 1 = task.
    groups : np.ndarray, shape (n_total_epochs,)
        Subject ID per epoch (for LOSO cross-validation).
    ch_names : list of str
        Channel names.
    mne_info : mne.Info
        MNE Info object (needed for topomaps).
    preprocessing_log : list of dict
        Per-subject preprocessing summary.
    """
    if config is None:
        config = CONFIG

    pairs = discover_subject_pairs(data_dir)

    if subject_range is not None:
        start, end = subject_range
        pairs = [(sid, r, t) for sid, r, t in pairs if start <= sid <= end]

    X_list, y_list, groups_list = [], [], []
    ch_names = None
    mne_info = None
    preprocessing_log = []

    for subject_id, rest_path, task_path in pairs:
        print(f"Processing Subject{subject_id:02d}...", end=" ")
        try:
            epochs_rest, epochs_task, subj_info = preprocess_subject(
                rest_path, task_path, config
            )
        except Exception as e:
            print(f"SKIPPED — {e}")
            preprocessing_log.append({
                'subject': subject_id, 'status': 'error', 'error': str(e)
            })
            continue

        if ch_names is None:
            ch_names = list(epochs_rest.ch_names)
            mne_info = epochs_rest.info.copy()

        rest_data = epochs_rest.get_data()  # (n_epochs, n_channels, n_times)
        task_data = epochs_task.get_data()

        X_list.append(rest_data)
        y_list.append(np.zeros(len(rest_data), dtype=int))
        groups_list.append(np.full(len(rest_data), subject_id, dtype=int))

        X_list.append(task_data)
        y_list.append(np.ones(len(task_data), dtype=int))
        groups_list.append(np.full(len(task_data), subject_id, dtype=int))

        log_entry = {
            'subject': subject_id,
            'status': 'ok',
            **subj_info,
        }
        preprocessing_log.append(log_entry)

        print(f"{subj_info['rest_epochs_clean']} rest, "
              f"{subj_info['task_epochs_clean']} task epochs "
              f"(rejected: {subj_info['rest_rejected']}+{subj_info['task_rejected']})")

    if not X_list:
        raise RuntimeError("No subjects were successfully preprocessed.")

    X = np.concatenate(X_list, axis=0)
    y = np.concatenate(y_list, axis=0)
    groups = np.concatenate(groups_list, axis=0)

    print(f"\nTotal: {len(X)} epochs ({(y == 0).sum()} rest, {(y == 1).sum()} task) "
          f"from {len([l for l in preprocessing_log if l['status'] == 'ok'])} subjects")

    return X, y, groups, ch_names, mne_info, preprocessing_log


# ========================================
# eeg_features.py
# ========================================

"""
eeg_features.py — Comprehensive EEG feature extraction for EEGMAT.

Five feature families extracted per epoch:
  A) Time-domain (shape statistics, Hjorth parameters)
  B) Frequency-domain (band power, workload ratios, spectral descriptors)
  C) Wavelet (DWT sub-band energy, entropy)
  D) Nonlinear / complexity (entropy, fractal dimension)
  E) Connectivity (correlation, PLV, coherence, graph metrics)

Usage:
    from eeg_features import build_feature_matrix
    df = build_feature_matrix(X, y, groups, sfreq=500, ch_names=ch_names)
    df.to_csv('eegmat_all_features.csv', index=False)
"""

import numpy as np
import pandas as pd
from scipy import signal, stats
from scipy.signal import welch, hilbert, butter, filtfilt, coherence as sp_coherence
import warnings

# Optional imports — gracefully handle if missing
try:
    import antropy as ant
    HAS_ANTROPY = True
except ImportError:
    HAS_ANTROPY = False

try:
    import pywt
    HAS_PYWT = True
except ImportError:
    HAS_PYWT = False

try:
    import networkx as nx
    HAS_NX = True
except ImportError:
    HAS_NX = False

try:
    from tqdm import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    def tqdm(x, **kwargs):
        return x

# ── Band definitions ───────────────────────────────────────────────────────
BAND_DEFS = {
    'delta': (1, 4),
    'theta': (4, 8),
    'alpha': (8, 13),
    'beta':  (13, 30),
    'gamma': (30, 45),
}

# Connectivity pairs — key inter-regional pairs from published EEG studies
CONNECTIVITY_PAIRS = [
    ('Fp1', 'Fp2'),   # frontal symmetry
    ('F3', 'F4'),     # frontal symmetry
    ('C3', 'C4'),     # central symmetry
    ('P3', 'P4'),     # parietal symmetry
    ('O1', 'O2'),     # occipital symmetry
    ('F3', 'P3'),     # fronto-parietal left
    ('F4', 'P4'),     # fronto-parietal right
    ('Fp1', 'O1'),    # antero-posterior left
    ('Fp2', 'O2'),    # antero-posterior right
    ('Fz', 'Pz'),     # midline fronto-parietal
]

# ══════════════════════════════════════════════════════════════════════════
# A) TIME-DOMAIN FEATURES
# ══════════════════════════════════════════════════════════════════════════

def time_domain_features(sig):
    """
    Basic shape statistics of the raw waveform.

    Parameters
    ----------
    sig : np.ndarray, shape (n_times,)
        Single-channel epoch signal.

    Returns
    -------
    feats : dict
        Time-domain feature names → values.
    """
    feats = {}

    # Central tendency and variability
    feats['mean'] = np.mean(sig)
    feats['std'] = np.std(sig)
    feats['variance'] = np.var(sig)
    feats['skewness'] = float(stats.skew(sig))
    feats['kurtosis'] = float(stats.kurtosis(sig))

    # Signal magnitude
    feats['rms'] = np.sqrt(np.mean(sig ** 2))
    feats['peak_to_peak'] = np.ptp(sig)

    # Zero-crossing rate (rough oscillation frequency proxy)
    zero_crossings = np.sum(np.diff(np.sign(sig)) != 0)
    feats['zero_crossing_rate'] = zero_crossings / len(sig)

    # Line length (total up-and-down distance → arousal proxy)
    feats['line_length'] = np.sum(np.abs(np.diff(sig)))

    # Hjorth parameters
    d1 = np.diff(sig)
    d2 = np.diff(d1)
    std_sig = np.std(sig) + 1e-10
    std_d1 = np.std(d1) + 1e-10

    feats['hjorth_activity'] = np.var(sig)
    feats['hjorth_mobility'] = std_d1 / std_sig
    feats['hjorth_complexity'] = (np.std(d2) / std_d1) / (feats['hjorth_mobility'] + 1e-10)

    return feats


# ══════════════════════════════════════════════════════════════════════════
# B) FREQUENCY-DOMAIN FEATURES
# ══════════════════════════════════════════════════════════════════════════

def frequency_domain_features(sig, sfreq):
    """
    Band power, workload ratios, and spectral descriptors via Welch PSD.

    Parameters
    ----------
    sig : np.ndarray, shape (n_times,)
    sfreq : float

    Returns
    -------
    feats : dict
    """
    feats = {}

    # Welch PSD
    nperseg = min(256, len(sig))
    freqs, psd = welch(sig, fs=sfreq, nperseg=nperseg)

    # Use np.trapezoid if available (NumPy 2.0+), else np.trapz
    _trapz = getattr(np, 'trapezoid', None) or np.trapz
    total_power = _trapz(psd, freqs) + 1e-20

    # Per-band power
    band_power = {}
    for band_name, (low, high) in BAND_DEFS.items():
        mask = (freqs >= low) & (freqs <= high)
        if mask.any():
            bp = _trapz(psd[mask], freqs[mask])
        else:
            bp = 0.0
        band_power[band_name] = bp

        feats[f'{band_name}_power'] = bp
        feats[f'{band_name}_power_log'] = np.log(bp + 1e-20)
        feats[f'{band_name}_relative_power'] = bp / total_power

    # Workload ratios (published cognitive-load indices)
    alpha = band_power.get('alpha', 1e-10) + 1e-10
    theta = band_power.get('theta', 1e-10) + 1e-10
    beta = band_power.get('beta', 1e-10) + 1e-10

    feats['engagement_index'] = beta / (alpha + theta)
    feats['theta_alpha_ratio'] = theta / alpha
    feats['theta_beta_ratio'] = theta / beta
    feats['alpha_theta_ratio'] = alpha / theta

    # 1/f slope (aperiodic exponent)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        if len(freqs) > 1:
            log_freqs = np.log(freqs[1:])
            log_psd = np.log(psd[1:] + 1e-20)
            slope, _ = np.polyfit(log_freqs, log_psd, 1)
            feats['spectral_slope'] = slope
        else:
            feats['spectral_slope'] = 0.0

    # Peak frequency (dominant oscillation)
    feats['peak_frequency'] = freqs[np.argmax(psd)]

    # Spectral edge frequencies (50%, 75%, 95%)
    cumulative = np.cumsum(psd)
    cumulative /= cumulative[-1] + 1e-20
    for pct in [0.50, 0.75, 0.95]:
        idx = np.searchsorted(cumulative, pct)
        feats[f'spectral_edge_{int(pct * 100)}'] = freqs[min(idx, len(freqs) - 1)]

    # Spectral entropy
    psd_norm = psd / (psd.sum() + 1e-20)
    psd_pos = psd_norm[psd_norm > 0]
    feats['spectral_entropy'] = -np.sum(psd_pos * np.log2(psd_pos))

    return feats


# ══════════════════════════════════════════════════════════════════════════
# C) WAVELET FEATURES
# ══════════════════════════════════════════════════════════════════════════

def wavelet_features(sig, wavelet='db4', level=5):
    """
    Discrete Wavelet Transform sub-band energies and entropy.

    Parameters
    ----------
    sig : np.ndarray, shape (n_times,)
    wavelet : str
        Wavelet family (default: Daubechies-4).
    level : int
        Decomposition level.

    Returns
    -------
    feats : dict
    """
    if not HAS_PYWT:
        return {}

    feats = {}
    coeffs = pywt.wavedec(sig, wavelet, level=level)

    energies = np.array([np.sum(c ** 2) for c in coeffs])
    total_energy = energies.sum() + 1e-20
    rel_energies = energies / total_energy

    for i, e in enumerate(rel_energies):
        feats[f'wavelet_energy_L{i}'] = energies[i]
        feats[f'wavelet_relative_energy_L{i}'] = e

    # Wavelet entropy
    p = rel_energies[rel_energies > 0]
    feats['wavelet_entropy'] = -np.sum(p * np.log2(p))

    return feats


# ══════════════════════════════════════════════════════════════════════════
# D) NONLINEAR / COMPLEXITY FEATURES
# ══════════════════════════════════════════════════════════════════════════

def nonlinear_features(sig):
    """
    Entropy and fractal dimension features measuring signal complexity.

    Parameters
    ----------
    sig : np.ndarray, shape (n_times,)

    Returns
    -------
    feats : dict
    """
    feats = {}

    if not HAS_ANTROPY:
        return feats

    # Permutation entropy — unpredictability of up/down patterns
    try:
        feats['permutation_entropy'] = ant.perm_entropy(sig, normalize=True)
    except Exception:
        feats['permutation_entropy'] = np.nan

    # SVD entropy — number of independent variation directions
    try:
        feats['svd_entropy'] = ant.svd_entropy(sig, normalize=True)
    except Exception:
        feats['svd_entropy'] = np.nan

    # Sample entropy — pattern repetitiveness
    try:
        feats['sample_entropy'] = ant.sample_entropy(sig)
    except Exception:
        feats['sample_entropy'] = np.nan

    # Approximate entropy
    try:
        feats['approximate_entropy'] = ant.app_entropy(sig)
    except Exception:
        feats['approximate_entropy'] = np.nan

    # Lempel-Ziv complexity
    try:
        feats['lz_complexity'] = ant.lziv_complexity(sig > np.median(sig), normalize=True)
    except Exception:
        feats['lz_complexity'] = np.nan

    # Fractal dimensions
    try:
        feats['higuchi_fd'] = ant.higuchi_fd(sig)
    except Exception:
        feats['higuchi_fd'] = np.nan

    try:
        feats['petrosian_fd'] = ant.petrosian_fd(sig)
    except Exception:
        feats['petrosian_fd'] = np.nan

    try:
        feats['katz_fd'] = ant.katz_fd(sig)
    except Exception:
        feats['katz_fd'] = np.nan

    # Detrended fluctuation analysis
    try:
        feats['detrended_fluctuation'] = ant.detrended_fluctuation(sig)
    except Exception:
        feats['detrended_fluctuation'] = np.nan

    # Number of zero crossings
    feats['num_zero_crossings'] = np.sum(np.diff(np.sign(sig)) != 0)

    return feats


# ══════════════════════════════════════════════════════════════════════════
# E) CONNECTIVITY FEATURES
# ══════════════════════════════════════════════════════════════════════════

def _phase_locking_value(sig1, sig2):
    """Phase Locking Value between two signals."""
    phase1 = np.angle(hilbert(sig1))
    phase2 = np.angle(hilbert(sig2))
    plv = np.abs(np.mean(np.exp(1j * (phase1 - phase2))))
    return plv


def _band_coherence(sig1, sig2, sfreq, band):
    """Mean coherence within a frequency band."""
    low, high = band
    nperseg = min(256, len(sig1))
    freqs, coh = sp_coherence(sig1, sig2, fs=sfreq, nperseg=nperseg)
    mask = (freqs >= low) & (freqs <= high)
    if mask.any():
        return coh[mask].mean()
    return np.nan


def connectivity_features(epoch, ch_names, sfreq, pairs=None):
    """
    Cross-channel connectivity: correlation, PLV, coherence.

    Parameters
    ----------
    epoch : np.ndarray, shape (n_channels, n_times)
        Single epoch, all channels.
    ch_names : list of str
        Channel names matching epoch's first dimension.
    sfreq : float
        Sampling frequency.
    pairs : list of (str, str), optional
        Channel pairs to compute connectivity for.

    Returns
    -------
    feats : dict
    """
    feats = {}

    if pairs is None:
        pairs = CONNECTIVITY_PAIRS

    # Build channel name → index map
    ch_idx = {ch: i for i, ch in enumerate(ch_names)}

    # Full correlation matrix for graph metrics
    corr_matrix = np.corrcoef(epoch)

    # Graph-theory metrics on the correlation matrix
    if HAS_NX:
        try:
            G = nx.from_numpy_array(np.abs(corr_matrix))
            feats['graph_clustering_coeff'] = nx.average_clustering(G, weight='weight')
            if nx.is_connected(G):
                feats['graph_avg_path_length'] = nx.average_shortest_path_length(G, weight='weight')
            else:
                feats['graph_avg_path_length'] = np.nan
        except Exception:
            feats['graph_clustering_coeff'] = np.nan
            feats['graph_avg_path_length'] = np.nan

    # Per-pair features
    for ch1_name, ch2_name in pairs:
        if ch1_name not in ch_idx or ch2_name not in ch_idx:
            continue

        i, j = ch_idx[ch1_name], ch_idx[ch2_name]
        pair_label = f'{ch1_name}_{ch2_name}'
        sig1, sig2 = epoch[i], epoch[j]

        # Pearson correlation
        feats[f'corr_{pair_label}'] = corr_matrix[i, j]

        # Phase Locking Value
        feats[f'plv_{pair_label}'] = _phase_locking_value(sig1, sig2)

        # Theta coherence
        feats[f'coherence_theta_{pair_label}'] = _band_coherence(
            sig1, sig2, sfreq, BAND_DEFS['theta']
        )

        # Alpha coherence
        feats[f'coherence_alpha_{pair_label}'] = _band_coherence(
            sig1, sig2, sfreq, BAND_DEFS['alpha']
        )

    return feats


# ══════════════════════════════════════════════════════════════════════════
# ORCHESTRATOR: combine all features for one epoch
# ══════════════════════════════════════════════════════════════════════════

def extract_all_features(epoch, sfreq, ch_names, include_nonlinear=True,
                         include_connectivity=True):
    """
    Extract ALL features for a single epoch.

    Parameters
    ----------
    epoch : np.ndarray, shape (n_channels, n_times)
        Raw-amplitude EEG data for one epoch.
    sfreq : float
        Sampling frequency.
    ch_names : list of str
        Channel names.
    include_nonlinear : bool
        If False, skip entropy/fractal features (faster).
    include_connectivity : bool
        If False, skip cross-channel features (faster).

    Returns
    -------
    feats : dict
        Flat dictionary of all features, keyed as 'channel_feature'.
    """
    feats = {}

    # Per-channel features (A, B, C, D)
    for ch_idx, ch_name in enumerate(ch_names):
        sig = epoch[ch_idx]

        # A) Time-domain
        for k, v in time_domain_features(sig).items():
            feats[f'{ch_name}_{k}'] = v

        # B) Frequency-domain
        for k, v in frequency_domain_features(sig, sfreq).items():
            feats[f'{ch_name}_{k}'] = v

        # C) Wavelet
        for k, v in wavelet_features(sig).items():
            feats[f'{ch_name}_{k}'] = v

        # D) Nonlinear
        if include_nonlinear:
            for k, v in nonlinear_features(sig).items():
                feats[f'{ch_name}_{k}'] = v

    # E) Connectivity (cross-channel)
    if include_connectivity:
        for k, v in connectivity_features(epoch, ch_names, sfreq).items():
            feats[k] = v

    return feats


# ══════════════════════════════════════════════════════════════════════════
# BUILD FEATURE MATRIX for the whole dataset
# ══════════════════════════════════════════════════════════════════════════

def build_feature_matrix(X, y, groups, sfreq, ch_names,
                         include_nonlinear=True, include_connectivity=True):
    """
    Build a feature DataFrame from the entire preprocessed dataset.

    Parameters
    ----------
    X : np.ndarray, shape (n_epochs, n_channels, n_times)
        Raw-amplitude epoch data (NOT z-scored).
    y : np.ndarray, shape (n_epochs,)
        Labels: 0 = rest, 1 = task.
    groups : np.ndarray, shape (n_epochs,)
        Subject IDs (for LOSO cross-validation).
    sfreq : float
        Sampling frequency.
    ch_names : list of str
        Channel names.
    include_nonlinear : bool
        If False, skip entropy/fractal features (faster).
    include_connectivity : bool
        If False, skip cross-channel features (faster).

    Returns
    -------
    df : pd.DataFrame
        One row per epoch, columns = features + 'label' + 'subject'.
    """
    n_epochs = len(X)
    all_features = []

    iterator = tqdm(range(n_epochs), desc="Extracting features",
                    unit="epoch") if HAS_TQDM else range(n_epochs)

    for i in iterator:
        feats = extract_all_features(
            X[i], sfreq, ch_names,
            include_nonlinear=include_nonlinear,
            include_connectivity=include_connectivity,
        )
        feats['label'] = int(y[i])
        feats['subject'] = int(groups[i])
        all_features.append(feats)

    df = pd.DataFrame(all_features)

    # Replace inf/nan with 0 for ML compatibility
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    n_features = len(df.columns) - 2  # exclude label and subject
    print(f"\nFeature matrix: {len(df)} epochs × {n_features} features")
    print(f"  NaN count: {df.isna().sum().sum()} (will be imputed during classification)")

    return df


# ========================================
# eeg_topomaps.py
# ========================================

"""
eeg_topomaps.py — EEG topomap generator for EEGMAT (rest vs. task).

Shows band power topographically so you can visually confirm e.g.
"this subject's frontal lobe engaged more during the task."

IMPORTANT: feed this RAW (not z-scored) epoch amplitude data. Per-epoch
z-scoring destroys the absolute power differences a topomap needs to show.

Usage:
    from eeg_topomaps import plot_subject_topomap, plot_group_topomaps

    # Single subject
    diff = plot_subject_topomap(mne_info, rest_epochs, task_epochs,
                                 band_name='theta', subject_id=3, save_dir='output/')

    # All subjects — grand average
    plot_group_topomaps(mne_info, X, y, groups, save_dir='output/')
"""

import os
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for saving figures
import matplotlib.pyplot as plt
from scipy.signal import welch
import mne


# ── Region definitions (10-20 system) ──────────────────────────────────────
REGIONS = {
    'frontal':  ['Fp1', 'Fp2', 'F3', 'F4', 'Fz', 'F7', 'F8'],
    'central':  ['C3', 'C4', 'Cz'],
    'temporal': ['T3', 'T4', 'T5', 'T6'],
    'parietal': ['P3', 'P4', 'Pz'],
    'occipital': ['O1', 'O2'],
}

BAND_DEFS = {
    'delta': (1, 4),
    'theta': (4, 8),
    'alpha': (8, 13),
    'beta':  (13, 30),
    'gamma': (30, 45),
}


def compute_band_power_per_channel(epochs_array, sfreq, band):
    """
    Compute log band power for each channel, for each epoch.

    Parameters
    ----------
    epochs_array : np.ndarray, shape (n_epochs, n_channels, n_times)
        RAW amplitude epoch data (NOT z-scored).
    sfreq : float
        Sampling frequency in Hz.
    band : tuple of (low_hz, high_hz)
        Frequency band boundaries.

    Returns
    -------
    power : np.ndarray, shape (n_epochs, n_channels)
        Log-transformed band power values.
    """
    low, high = band
    n_epochs, n_channels, n_times = epochs_array.shape
    power = np.zeros((n_epochs, n_channels))
    nperseg = min(256, n_times)

    for e in range(n_epochs):
        for ch in range(n_channels):
            freqs, psd = welch(epochs_array[e, ch], fs=sfreq, nperseg=nperseg)
            mask = (freqs >= low) & (freqs <= high)
            if mask.any():
                power[e, ch] = np.log(psd[mask].mean() + 1e-20)
            else:
                power[e, ch] = np.nan

    return power


def plot_subject_topomap(info, rest_power, task_power, band_name,
                         subject_id, save_dir=None):
    """
    Plot 3-panel topomap: rest | task | difference for one subject.

    Parameters
    ----------
    info : mne.Info
        MNE Info with channel locations.
    rest_power : np.ndarray, shape (n_channels,)
        Mean band power during rest (averaged over epochs).
    task_power : np.ndarray, shape (n_channels,)
        Mean band power during task (averaged over epochs).
    band_name : str
        Name of the frequency band (e.g., 'theta').
    subject_id : int
        Subject number for labeling.
    save_dir : str, optional
        Directory to save the figure. If None, doesn't save.

    Returns
    -------
    diff : np.ndarray, shape (n_channels,)
        Task minus rest power.
    """
    diff = task_power - rest_power

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Use consistent vmin/vmax across rest and task for fair comparison
    vmin = min(rest_power.min(), task_power.min())
    vmax = max(rest_power.max(), task_power.max())

    im1, _ = mne.viz.plot_topomap(rest_power, info, axes=axes[0], show=False,
                                   cmap='RdBu_r', vlim=(vmin, vmax))
    axes[0].set_title(f'Rest — {band_name}')

    im2, _ = mne.viz.plot_topomap(task_power, info, axes=axes[1], show=False,
                                   cmap='RdBu_r', vlim=(vmin, vmax))
    axes[1].set_title(f'Task — {band_name}')

    im3, _ = mne.viz.plot_topomap(diff, info, axes=axes[2], show=False,
                                   cmap='RdBu_r')
    axes[2].set_title(f'Task − Rest — {band_name}')
    fig.colorbar(im3, ax=axes[2], shrink=0.7)

    fig.suptitle(f'Subject {subject_id:02d} — {band_name.capitalize()} Band Power',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        path = os.path.join(save_dir, f'subject_{subject_id:02d}_{band_name}_topomap.png')
        plt.savefig(path, dpi=150, bbox_inches='tight')
        print(f"  Saved: {path}")

    plt.close(fig)
    return diff


def frontal_lobe_summary(diff, ch_names, frontal_channels=None):
    """
    Compare frontal vs. other regions' power change (task − rest).

    Parameters
    ----------
    diff : np.ndarray, shape (n_channels,)
        Task minus rest power.
    ch_names : list of str
        Channel names matching diff.
    frontal_channels : list of str, optional
        Override default frontal channel list.

    Returns
    -------
    summary : dict with 'frontal', 'other', 'frontal_greater' keys.
    """
    if frontal_channels is None:
        frontal_channels = REGIONS['frontal']

    idx = [i for i, ch in enumerate(ch_names) if ch in frontal_channels]
    if not idx:
        print(f"  Warning: no frontal channels found in {ch_names}")
        return None

    frontal_diff = diff[idx].mean()
    other_idx = [i for i in range(len(diff)) if i not in idx]
    other_diff = diff[other_idx].mean() if other_idx else 0.0

    return {
        'frontal': frontal_diff,
        'other': other_diff,
        'frontal_greater': frontal_diff > other_diff,
    }


def region_summary(diff, ch_names):
    """
    Compute mean power change per brain region.

    Returns
    -------
    region_diffs : dict mapping region name → mean (task − rest) power.
    """
    result = {}
    for region, channels in REGIONS.items():
        idx = [i for i, ch in enumerate(ch_names) if ch in channels]
        if idx:
            result[region] = diff[idx].mean()
    return result


def plot_group_topomaps(info, X, y, groups, sfreq=500, bands=None, save_dir=None):
    """
    Plot grand-average topomaps across ALL subjects.

    Parameters
    ----------
    info : mne.Info
        MNE Info with channel locations.
    X : np.ndarray, shape (n_epochs, n_channels, n_times)
        Raw amplitude epochs.
    y : np.ndarray
        Labels (0=rest, 1=task).
    groups : np.ndarray
        Subject IDs.
    sfreq : float
        Sampling frequency.
    bands : list of str, optional
        Which bands to plot. Default: ['theta', 'alpha', 'beta'].
    save_dir : str, optional
        Directory to save figures.

    Returns
    -------
    results : dict mapping band_name → (rest_power_mean, task_power_mean, diff)
    """
    if bands is None:
        bands = ['theta', 'alpha', 'beta']

    results = {}

    for band_name in bands:
        band = BAND_DEFS[band_name]
        print(f"Computing grand-average {band_name} topomaps...")

        rest_mask = y == 0
        task_mask = y == 1

        rest_power = compute_band_power_per_channel(X[rest_mask], sfreq, band).mean(axis=0)
        task_power = compute_band_power_per_channel(X[task_mask], sfreq, band).mean(axis=0)

        diff = plot_subject_topomap(
            info, rest_power, task_power,
            band_name=band_name,
            subject_id='group_avg',
            save_dir=save_dir,
        )

        results[band_name] = (rest_power, task_power, diff)

    return results


def run_all_subjects_frontal_report(X, y, groups, ch_names, sfreq=500, band='theta'):
    """
    Compute frontal engagement summary for every subject.

    Returns
    -------
    report : list of dict with keys: subject, frontal, other, frontal_greater
    """
    band_range = BAND_DEFS[band]
    unique_subjects = np.unique(groups)
    report = []

    for sid in unique_subjects:
        rest_mask = (groups == sid) & (y == 0)
        task_mask = (groups == sid) & (y == 1)

        if rest_mask.sum() == 0 or task_mask.sum() == 0:
            continue

        rest_power = compute_band_power_per_channel(
            X[rest_mask], sfreq, band_range
        ).mean(axis=0)
        task_power = compute_band_power_per_channel(
            X[task_mask], sfreq, band_range
        ).mean(axis=0)

        diff = task_power - rest_power
        summary = frontal_lobe_summary(diff, ch_names)
        if summary:
            summary['subject'] = sid
            report.append(summary)

    # Print summary
    n_frontal = sum(1 for r in report if r['frontal_greater'])
    print(f"\nFrontal {band} engagement report:")
    print(f"  {n_frontal}/{len(report)} subjects show greater frontal than "
          f"non-frontal {band} power increase during task")

    return report


# ========================================
# eegmat_classify.py
# ========================================

"""
eegmat_classify.py — Classification pipeline for EEGMAT (rest vs. task).

Implements Leave-One-Subject-Out (LOSO) cross-validation with multiple
classifiers, feature importance analysis, and result visualization.

Usage:
    from eegmat_classify import run_classification

    results = run_classification(df, save_dir='output/')
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, roc_auc_score,
)
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)


# ── Classifier definitions ─────────────────────────────────────────────────

def get_classifiers():
    """Return dict of classifier name → sklearn estimator."""
    return {
        'Random Forest': RandomForestClassifier(
            n_estimators=200,
            max_depth=20,
            min_samples_split=5,
            random_state=42,
            n_jobs=-1,
        ),
        'Logistic Regression': LogisticRegression(
            max_iter=2000,
            C=1.0,
            random_state=42,
            n_jobs=-1,
        ),
        'SVM (RBF)': SVC(
            kernel='rbf',
            C=10.0,
            gamma='scale',
            probability=True,
            random_state=42,
        ),
    }


# ── LOSO Cross-Validation ──────────────────────────────────────────────────

def loso_cross_validation(df, classifier, classifier_name='Classifier'):
    """
    Leave-One-Subject-Out cross-validation.

    Parameters
    ----------
    df : pd.DataFrame
        Feature matrix with 'label' and 'subject' columns.
    classifier : sklearn estimator
        Classifier to evaluate.
    classifier_name : str
        Name for logging.

    Returns
    -------
    result : dict with keys:
        'accuracy', 'f1', 'auc', 'per_subject_accuracy',
        'y_true', 'y_pred', 'y_prob', 'feature_importances'
    """
    feature_cols = [c for c in df.columns if c not in ('label', 'subject')]
    X = df[feature_cols].values
    y = df['label'].values
    groups = df['subject'].values

    logo = LeaveOneGroupOut()

    y_true_all, y_pred_all, y_prob_all = [], [], []
    per_subject = {}
    importances_sum = np.zeros(len(feature_cols))
    n_folds = 0

    for train_idx, test_idx in logo.split(X, y, groups):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        test_subject = groups[test_idx][0]

        # Impute NaN, then scale — fit on TRAIN only
        imputer = SimpleImputer(strategy='median')
        X_train = imputer.fit_transform(X_train)
        X_test = imputer.transform(X_test)

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # Fit and predict
        from sklearn.base import clone
        clf = clone(classifier)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        # Probabilities for AUC
        if hasattr(clf, 'predict_proba'):
            y_prob = clf.predict_proba(X_test)[:, 1]
        else:
            y_prob = clf.decision_function(X_test)

        y_true_all.extend(y_test)
        y_pred_all.extend(y_pred)
        y_prob_all.extend(y_prob)

        fold_acc = accuracy_score(y_test, y_pred)
        per_subject[int(test_subject)] = fold_acc

        # Accumulate feature importances (RF has feature_importances_)
        if hasattr(clf, 'feature_importances_'):
            importances_sum += clf.feature_importances_
        elif hasattr(clf, 'coef_'):
            importances_sum += np.abs(clf.coef_[0])

        n_folds += 1

    y_true_all = np.array(y_true_all)
    y_pred_all = np.array(y_pred_all)
    y_prob_all = np.array(y_prob_all)

    overall_acc = accuracy_score(y_true_all, y_pred_all)
    overall_f1 = f1_score(y_true_all, y_pred_all, average='weighted')

    try:
        overall_auc = roc_auc_score(y_true_all, y_prob_all)
    except ValueError:
        overall_auc = np.nan

    avg_importances = importances_sum / max(n_folds, 1)

    print(f"\n{classifier_name} (LOSO):")
    print(f"  Accuracy: {overall_acc:.4f}")
    print(f"  F1:       {overall_f1:.4f}")
    print(f"  AUC:      {overall_auc:.4f}")
    print(f"  Per-subject range: [{min(per_subject.values()):.2f}, "
          f"{max(per_subject.values()):.2f}]")

    return {
        'classifier': classifier_name,
        'accuracy': overall_acc,
        'f1': overall_f1,
        'auc': overall_auc,
        'per_subject_accuracy': per_subject,
        'y_true': y_true_all,
        'y_pred': y_pred_all,
        'y_prob': y_prob_all,
        'feature_names': feature_cols,
        'feature_importances': avg_importances,
        'confusion_matrix': confusion_matrix(y_true_all, y_pred_all),
        'classification_report': classification_report(y_true_all, y_pred_all,
                                                        target_names=['Rest', 'Task']),
    }


# ── Plotting ───────────────────────────────────────────────────────────────

def plot_confusion_matrix(result, save_dir=None):
    """Plot and save confusion matrix heatmap."""
    fig, ax = plt.subplots(figsize=(6, 5))
    cm = result['confusion_matrix']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Rest', 'Task'],
                yticklabels=['Rest', 'Task'], ax=ax)
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('True', fontsize=12)
    ax.set_title(f"{result['classifier']} — Confusion Matrix\n"
                 f"Accuracy: {result['accuracy']:.3f}, F1: {result['f1']:.3f}",
                 fontsize=13)
    plt.tight_layout()

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        name = result['classifier'].replace(' ', '_').replace('(', '').replace(')', '')
        path = os.path.join(save_dir, f'{name}_confusion_matrix.png')
        plt.savefig(path, dpi=150, bbox_inches='tight')
        print(f"  Saved: {path}")

    plt.close(fig)


def plot_per_subject_accuracy(result, save_dir=None):
    """Bar chart of per-subject accuracy."""
    psa = result['per_subject_accuracy']
    subjects = sorted(psa.keys())
    accs = [psa[s] for s in subjects]

    fig, ax = plt.subplots(figsize=(14, 5))
    colors = ['#e74c3c' if a < 0.5 else '#2ecc71' if a >= 0.8 else '#f39c12'
              for a in accs]
    ax.bar([f'S{s:02d}' for s in subjects], accs, color=colors, edgecolor='white')
    ax.axhline(y=result['accuracy'], color='#3498db', linestyle='--', linewidth=2,
               label=f'Mean: {result["accuracy"]:.3f}')
    ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_xlabel('Subject', fontsize=12)
    ax.set_title(f'{result["classifier"]} — Per-Subject LOSO Accuracy', fontsize=13)
    ax.legend(fontsize=11)
    ax.set_ylim(0, 1.05)
    plt.xticks(rotation=45, fontsize=8)
    plt.tight_layout()

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        name = result['classifier'].replace(' ', '_').replace('(', '').replace(')', '')
        path = os.path.join(save_dir, f'{name}_per_subject_accuracy.png')
        plt.savefig(path, dpi=150, bbox_inches='tight')
        print(f"  Saved: {path}")

    plt.close(fig)


def plot_feature_importance(result, top_n=30, save_dir=None):
    """Horizontal bar chart of top feature importances."""
    importances = result['feature_importances']
    names = result['feature_names']

    if importances.sum() == 0:
        print("  No feature importances available for this classifier.")
        return

    sorted_idx = np.argsort(importances)[::-1][:top_n]
    top_names = [names[i] for i in sorted_idx]
    top_vals = importances[sorted_idx]

    fig, ax = plt.subplots(figsize=(10, max(6, top_n * 0.3)))
    y_pos = np.arange(len(top_names))
    ax.barh(y_pos, top_vals[::-1], color='#3498db', edgecolor='white')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(top_names[::-1], fontsize=9)
    ax.set_xlabel('Importance', fontsize=12)
    ax.set_title(f'{result["classifier"]} — Top {top_n} Feature Importances', fontsize=13)
    plt.tight_layout()

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        name = result['classifier'].replace(' ', '_').replace('(', '').replace(')', '')
        path = os.path.join(save_dir, f'{name}_feature_importance.png')
        plt.savefig(path, dpi=150, bbox_inches='tight')
        print(f"  Saved: {path}")

    plt.close(fig)


def plot_classifier_comparison(all_results, save_dir=None):
    """Compare all classifiers' accuracy, F1, AUC in a grouped bar chart."""
    names = [r['classifier'] for r in all_results]
    metrics = ['accuracy', 'f1', 'auc']
    metric_labels = ['Accuracy', 'F1 Score', 'AUC']

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(names))
    width = 0.25

    for i, (m, label) in enumerate(zip(metrics, metric_labels)):
        vals = [r[m] for r in all_results]
        bars = ax.bar(x + i * width, vals, width, label=label, edgecolor='white')
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=9)

    ax.set_ylabel('Score', fontsize=12)
    ax.set_title('Classifier Comparison — LOSO Cross-Validation', fontsize=14)
    ax.set_xticks(x + width)
    ax.set_xticklabels(names, fontsize=11)
    ax.legend(fontsize=11)
    ax.set_ylim(0, 1.15)
    plt.tight_layout()

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        path = os.path.join(save_dir, 'classifier_comparison.png')
        plt.savefig(path, dpi=150, bbox_inches='tight')
        print(f"  Saved: {path}")

    plt.close(fig)


# ── Main entry point ───────────────────────────────────────────────────────

def run_classification(df, save_dir=None, classifiers=None):
    """
    Run LOSO cross-validation with multiple classifiers.

    Parameters
    ----------
    df : pd.DataFrame
        Feature matrix from build_feature_matrix().
    save_dir : str, optional
        Directory to save plots.
    classifiers : dict, optional
        Override default classifiers.

    Returns
    -------
    all_results : list of dict
        Per-classifier results.
    """
    if classifiers is None:
        classifiers = get_classifiers()

    print(f"Dataset: {len(df)} epochs, "
          f"{len(df.columns) - 2} features, "
          f"{df['subject'].nunique()} subjects")
    print(f"Class balance: {(df['label'] == 0).sum()} rest, "
          f"{(df['label'] == 1).sum()} task\n")

    all_results = []

    for name, clf in classifiers.items():
        result = loso_cross_validation(df, clf, classifier_name=name)
        all_results.append(result)

        if save_dir:
            plot_confusion_matrix(result, save_dir)
            plot_per_subject_accuracy(result, save_dir)
            plot_feature_importance(result, save_dir=save_dir)

    if save_dir and len(all_results) > 1:
        plot_classifier_comparison(all_results, save_dir)

    # Print summary table
    print("\n" + "=" * 60)
    print("CLASSIFICATION SUMMARY (LOSO Cross-Validation)")
    print("=" * 60)
    print(f"{'Classifier':<25} {'Accuracy':>10} {'F1':>10} {'AUC':>10}")
    print("-" * 60)
    for r in all_results:
        print(f"{r['classifier']:<25} {r['accuracy']:>10.4f} {r['f1']:>10.4f} "
              f"{r['auc']:>10.4f}")
    print("=" * 60)

    return all_results


